In [1]:
# Install the KaggleHub package
!pip install -q kagglehub

In [2]:
# Authenticate KaggleHub with your Kaggle API token
import kagglehub

kagglehub.login()

Kaggle credentials set.
Kaggle credentials successfully validated.


In [3]:
# Download the competition files
path = kagglehub.competition_download("predictive-modelling-ds")

print("Path to competition files:", path)

100%|██████████| 5.15M/5.15M [00:01<00:00, 4.44MB/s]

Extracting files...


Path to competition files: /root/.cache/kagglehub/competitions/predictive-modelling-ds


In [4]:
# Load all competition datasets into separate DataFrames
import pandas as pd
import os
train = pd.read_csv(os.path.join(path, "train_videos.csv"))
test = pd.read_csv(os.path.join(path, "test_videos.csv"))
engagement = pd.read_csv(os.path.join(path, "engagement_daily.csv"))
creators = pd.read_csv(os.path.join(path, "creators_daily.csv"))
sample_submission = pd.read_csv(
    os.path.join(path, "sample_submission.csv")
)

print("Train:", train.shape)
print("Test:", test.shape)
print("Engagement:", engagement.shape)
print("Creators:", creators.shape)
print("Sample submission:", sample_submission.shape)

Train: (12000, 27)
Test: (3001, 26)
Engagement: (79489, 10)
Creators: (252166, 7)
Sample submission: (3001, 2)


# **كودي**

In [11]:
# Import libraries for feature engineering and modeling
import numpy as np
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor

In [12]:
# Convert date columns to datetime for historical filtering
train["create_date"] = pd.to_datetime(train["create_date"])
test["create_date"] = pd.to_datetime(test["create_date"])
engagement["date"] = pd.to_datetime(engagement["date"])
creators["date"] = pd.to_datetime(creators["date"])

In [13]:
# Keep only engagement observations from Day 0 through Day 5
eng_05 = engagement[
    engagement["days_since_post"].between(0, 5)
].copy()

print("Engagement rows used:", len(eng_05))

Engagement rows used: 79489


In [14]:
# Extract the latest available engagement values at Day 5
day5 = (
    eng_05.loc[
        eng_05["days_since_post"].eq(5),
        [
            "video_id",
            "play_count",
            "like_count",
            "comment_count",
            "share_count",
            "collect_count",
            "download_count",
            "whatsapp_share_count"
        ]
    ]
    .rename(
        columns={
            "play_count": "day5_plays",
            "like_count": "day5_likes",
            "comment_count": "day5_comments",
            "share_count": "day5_shares",
            "collect_count": "day5_collects",
            "download_count": "day5_downloads",
            "whatsapp_share_count": "day5_whatsapp_shares"
        }
    )
)

In [15]:
# Create summary statistics from the first six engagement days
eng_summary = (
    eng_05.groupby("video_id")
    .agg(
        views_mean_0_5=("play_count", "mean"),
        views_max_0_5=("play_count", "max"),
        views_std_0_5=("play_count", "std"),
        likes_mean_0_5=("like_count", "mean"),
        comments_mean_0_5=("comment_count", "mean"),
        shares_mean_0_5=("share_count", "mean"),
        collects_mean_0_5=("collect_count", "mean"),
        downloads_mean_0_5=("download_count", "mean"),
        whatsapp_mean_0_5=("whatsapp_share_count", "mean")
    )
    .reset_index()
)

In [16]:
# Extract Day 0 and Day 5 views to measure early growth
day0 = (
    eng_05.loc[
        eng_05["days_since_post"].eq(0),
        ["video_id", "play_count"]
    ]
    .rename(columns={"play_count": "day0_plays"})
)

growth = day0.merge(
    day5[["video_id", "day5_plays"]],
    on="video_id",
    how="outer"
)

growth["day0_plays"] = growth["day0_plays"].fillna(0)
growth["day5_plays"] = growth["day5_plays"].fillna(0)

growth["views_growth"] = (
    growth["day5_plays"] - growth["day0_plays"]
)

growth["views_growth_ratio"] = (
    growth["day5_plays"] /
    growth["day0_plays"].replace(0, np.nan)
)

growth["views_growth_ratio"] = (
    growth["views_growth_ratio"]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

In [17]:
# Create interaction-to-view ratios using Day 5 engagement
interaction_features = day5.copy()

views = interaction_features["day5_plays"].replace(0, np.nan)

interaction_features["like_view_ratio"] = (
    interaction_features["day5_likes"] / views
)

interaction_features["comment_view_ratio"] = (
    interaction_features["day5_comments"] / views
)

interaction_features["share_view_ratio"] = (
    interaction_features["day5_shares"] / views
)

interaction_features["collect_view_ratio"] = (
    interaction_features["day5_collects"] / views
)

interaction_features["download_view_ratio"] = (
    interaction_features["day5_downloads"] / views
)

interaction_features["whatsapp_share_view_ratio"] = (
    interaction_features["day5_whatsapp_shares"] / views
)

ratio_columns = [
    "like_view_ratio",
    "comment_view_ratio",
    "share_view_ratio",
    "collect_view_ratio",
    "download_view_ratio",
    "whatsapp_share_view_ratio"
]

interaction_features[ratio_columns] = (
    interaction_features[ratio_columns]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

In [19]:
# Create historical creator features for both training and test videos
all_videos = pd.concat(
    [
        train[["video_id", "author_id", "create_date"]],
        test[["video_id", "author_id", "create_date"]]
    ],
    ignore_index=True
)

all_videos["end_date"] = (
    all_videos["create_date"] +
    pd.Timedelta(days=5)
)

creator_history = all_videos.merge(
    creators,
    on="author_id",
    how="left"
)

creator_history = creator_history[
    creator_history["date"].between(
        creator_history["create_date"],
        creator_history["end_date"]
    )
]

creator_agg = (
    creator_history.groupby("video_id")
    .agg(
        creator_follower_mean=("follower_count", "mean"),
        creator_follower_max=("follower_count", "max"),
        creator_following_mean=("following_count", "mean"),
        creator_favorited_mean=("total_favorited", "mean"),
        creator_video_count_mean=("video_count", "mean"),
        creator_verified=("enterprise_verified", "max")
    )
    .reset_index()
)

In [37]:
# Rebuild train and test feature tables with one Day 5 views column
train_features = train.copy()
test_features = test.copy()

# Add basic time features
for df in [train_features, test_features]:
    df["create_hour"] = pd.to_datetime(
        df["create_time"]
    ).dt.hour

    df["create_weekday"] = pd.to_datetime(
        df["create_date"]
    ).dt.weekday

# Merge Day 5 engagement and interaction features
train_features = train_features.merge(
    interaction_features,
    on="video_id",
    how="left"
)

test_features = test_features.merge(
    interaction_features,
    on="video_id",
    how="left"
)

# Merge engagement summary features
train_features = train_features.merge(
    eng_summary,
    on="video_id",
    how="left"
)

test_features = test_features.merge(
    eng_summary,
    on="video_id",
    how="left"
)

# Merge only the Day 0 views and growth features
growth_for_merge = growth[
    [
        "video_id",
        "day0_plays",
        "views_growth",
        "views_growth_ratio"
    ]
]

train_features = train_features.merge(
    growth_for_merge,
    on="video_id",
    how="left"
)

test_features = test_features.merge(
    growth_for_merge,
    on="video_id",
    how="left"
)

# Merge historical creator features
train_features = train_features.merge(
    creator_agg,
    on="video_id",
    how="left"
)

test_features = test_features.merge(
    creator_agg,
    on="video_id",
    how="left"
)

print("Train shape:", train_features.shape)
print("Test shape:", test_features.shape)

Train shape: (12000, 60)
Test shape: (3001, 59)


In [39]:
# Fill missing numerical features without including the target
numeric_columns = [
    col
    for col in train_features.select_dtypes(include=np.number).columns
    if col != "target_day30_views"
]

train_features[numeric_columns] = (
    train_features[numeric_columns]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

test_features[numeric_columns] = (
    test_features[numeric_columns]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

In [45]:
# Convert video resolution values such as 540p into numeric values
train_features["ratio_numeric"] = (
    train_features["ratio"]
    .astype(str)
    .str.extract(r"(\d+)", expand=False)
    .astype(float)
)

test_features["ratio_numeric"] = (
    test_features["ratio"]
    .astype(str)
    .str.extract(r"(\d+)", expand=False)
    .astype(float)
)

In [46]:
# Use numeric resolution instead of the original text resolution
features = [
    "duration",
    "ratio_numeric",
    "is_english",
    "created_by_ai",
    "is_ads",
    "word_count",
    "emoji_count",
    "question_count",
    "hashtag_count",
    "speaking_rate",
    "anger",
    "joy",
    "surprise",
    "sadness",
    "disgust",
    "fear",
    "create_hour",
    "create_weekday",

    "day5_plays",
    "day5_likes",
    "day5_comments",
    "day5_shares",
    "day5_collects",
    "day5_downloads",
    "day5_whatsapp_shares",

    "views_mean_0_5",
    "views_max_0_5",
    "views_std_0_5",
    "likes_mean_0_5",
    "comments_mean_0_5",
    "shares_mean_0_5",
    "collects_mean_0_5",
    "downloads_mean_0_5",
    "whatsapp_mean_0_5",

    "views_growth",
    "views_growth_ratio",

    "like_view_ratio",
    "comment_view_ratio",
    "share_view_ratio",
    "collect_view_ratio",
    "download_view_ratio",
    "whatsapp_share_view_ratio",

    "creator_follower_mean",
    "creator_follower_max",
    "creator_following_mean",
    "creator_favorited_mean",
    "creator_video_count_mean",
    "creator_verified"
]

In [47]:
# Rebuild the training matrix after converting resolution to a numeric feature
X = train_features[features]
y = train_features["target_day30_views"]

print("X shape:", X.shape)
print("Non-numeric columns:", X.select_dtypes(exclude=np.number).columns.tolist())

X shape: (12000, 48)
Non-numeric columns: []


In [49]:
# Import the standard RMSE metric
from sklearn.metrics import mean_squared_error

In [101]:
# Define 5-fold cross-validation with a fixed random seed
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [85]:
rf_poisson_scores = []
rf_poisson_negative_predictions = []

for train_idx, valid_idx in kf.split(X):
    X_train = X.iloc[train_idx]
    X_valid = X.iloc[valid_idx]
    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    model = RandomForestRegressor(
        n_estimators=500,
        max_depth=None,
        min_samples_leaf=2,
        max_features=0.7,
        criterion="poisson",
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    valid_predictions = model.predict(X_valid)

    # Check whether the model produced any negative predictions
    rf_poisson_negative_predictions.append(
        np.sum(valid_predictions < 0)
    )

    valid_predictions = np.clip(valid_predictions, 0, None)

    rmse = np.sqrt(
        mean_squared_error(y_valid, valid_predictions)
    )

    rf_poisson_scores.append(rmse)

print("Random Forest Poisson CV RMSE:")
print([f"{score:,.2f}" for score in rf_poisson_scores])

print(f"Mean CV RMSE: {np.mean(rf_poisson_scores):,.2f}")

print("Negative predictions:", rf_poisson_negative_predictions)
print("Total negative predictions:", sum(rf_poisson_negative_predictions))

Random Forest Poisson CV RMSE:
['70,957.90', '98,812.56', '51,461.84', '200,378.13', '57,198.73']
Mean CV RMSE: 95,761.83
Negative predictions: [np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0)]
Total negative predictions: 0


In [93]:
# Train the final Random Forest Poisson model on all training data
final_rf_poisson = RandomForestRegressor(
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=2,
    max_features=0.7,
    criterion="poisson",
    random_state=42,
    n_jobs=-1
)

final_rf_poisson.fit(X, y)

# Generate predictions for the test set
test_predictions_rf_poisson = final_rf_poisson.predict(
    test_features[features]
)

# Ensure predictions are non-negative
test_predictions_rf_poisson = np.clip(
    test_predictions_rf_poisson,
    0,
    None
)

# Create submission using the required sample_submission structure
submission_rf_poisson = sample_submission.copy()
submission_rf_poisson["target_day30_views"] = test_predictions_rf_poisson

# Save the submission file
submission_rf_poisson.to_csv(
    "final_random_forest_poisson_submission.csv",
    index=False
)

# Check the final submission
print("Submission shape:", submission_rf_poisson.shape)
print("Columns:", submission_rf_poisson.columns.tolist())
print("Missing predictions:", submission_rf_poisson["target_day30_views"].isna().sum())
print("Negative predictions:", (submission_rf_poisson["target_day30_views"] < 0).sum())
print("Duplicate video IDs:", submission_rf_poisson["video_id"].duplicated().sum())

print("\nPrediction statistics:")
print(submission_rf_poisson["target_day30_views"].describe())

print("\nFirst 10 rows:")
print(submission_rf_poisson.head(10))

print("\nFile saved as:")
print("final_random_forest_poisson_submission.csv")

Submission shape: (3001, 2)
Columns: ['video_id', 'target_day30_views']
Missing predictions: 0
Negative predictions: 0
Duplicate video IDs: 0

Prediction statistics:
count    3.001000e+03
mean     2.674752e+04
std      1.963009e+05
min      1.275400e+01
25%      3.514518e+02
50%      7.933392e+02
75%      3.465389e+03
max      6.208182e+06
Name: target_day30_views, dtype: float64

First 10 rows:
              video_id  target_day30_views
0  7400582591771938090          242.772971
1  7403167206978374958          387.017072
2  7435124823820356906         2521.396593
3  7409800970143714606          294.358698
4  7410935177553333550          282.653987
5  7432809888033656095         2243.249644
6  7405432909278022955        58486.810121
7  7384996099955756331          332.543939
8  7421705386400517406       135230.911139
9  7414296612047965470          682.533348

File saved as:
final_random_forest_poisson_submission.csv
